# HW2: Optimizing a soft/stiff cantilever

You are given a mass-spring-damper (MSD) simulator of a 2-D cantilever beam made of
square cells. Every cell is either **soft** or **stiff**. Your job is to choose the
material of every cell so that the bent beam's centerline matches a target shape,
and to compare three optimizers:

1. random search
2. a genetic algorithm (evolutionary)
3. a gradient-based method (JAX differentiates through the whole simulation)

**How to use this notebook**

* **Google Colab:** first choose *File → Save a copy in Drive* and work in your copy.
  Then run the setup cell in section 2. It downloads the simulator from GitHub.
* **Your own computer (Jupyter or VS Code):** open this notebook inside the folder
  you cloned from GitHub (see `README.md`). The same setup cell finds the files there.
* Run the cells in order, top to bottom (`Shift + Enter` runs a cell and moves on).
  *Restart and Run All* also works at any time, even before you have written any
  code: unfinished parts print `TODO` instead of crashing.

> ⚠️ **Colab's local filesystem is temporary.** Save your notebook to Drive and
> download your final submission ZIP before ending the session.
* You only edit the cells marked **✏️ STUDENT TODO** (sections 9, 10 and 11), plus any
  analysis cells you add yourself. Do not edit the `.py` files: they are the
  simulator and the evaluation counting, and they are the same for everyone.
* Cells marked **⏱** take more than a few seconds. Their expected run time is given.
* The very first simulation compiles the simulator (a few seconds). Later calls are fast.

**Coming from MATLAB?** Indexing starts at 0 and `x[a:b]` excludes `b`;
`x[-1]` is the last element. `np.zeros(n)` is a 1-D array (no row/column
distinction), `@` is matrix multiplication, and `**` is the power operator.
Use one random generator per run, `rng = np.random.default_rng(seed)`, then
`rng.random(n)` or `rng.integers(0, 2, n)`, so every run is reproducible.

## 2. Setup and environment check

**Run this cell first, every session.** It finds the HW2 simulator files (the
"backend") and checks the Python packages.

* On **Colab** it downloads the backend from GitHub into the folder `hw2_backend/`
  (or updates it if it is already there). It never touches your notebook or your
  `submission/` folder.
* On **your own computer** it uses the files next to this notebook and downloads
  nothing.

Optional, Colab only: set `USE_DRIVE = True` to keep results and submission files in
your Google Drive (folder `HW2_outputs`). They then survive a disconnect, so a long
experiment can be resumed in a later session. Colab will ask for permission to
access your Drive.

In [ ]:
# ---- Course staff: the backend repository is set here, in one place. ----
GITHUB_REPO = "https://github.com/iamtanay7/AMDS-HW2.git"
BACKEND_DIR = "hw2_backend"     # Colab: folder the backend is downloaded into
USE_DRIVE = False               # Colab: True = keep results/ and submission/ in Google Drive

import os, re, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

try:
    import google.colab         # this module only exists on Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 1. Find (Colab: download or update) the backend.
if IN_COLAB:
    if os.path.isdir(os.path.join(BACKEND_DIR, ".git")):
        pull = subprocess.run(["git", "-C", BACKEND_DIR, "pull", "--ff-only", "-q"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("Could not update the backend, so the existing copy is used.\n"
                  "(This happens if files inside hw2_backend/ were edited. Your notebook and\n"
                  "submission/ are not affected.) git said:\n" + pull.stderr.strip())
    elif not os.path.exists(BACKEND_DIR):
        clone = subprocess.run(["git", "clone", "-q", "--depth", "1", GITHUB_REPO, BACKEND_DIR],
                               capture_output=True, text=True)
        if clone.returncode != 0:
            raise RuntimeError(f"Could not download {GITHUB_REPO}:\n{clone.stderr}")
    BACKEND = os.path.abspath(BACKEND_DIR)
    OUTPUT_DIR = "/content/drive/MyDrive/HW2_outputs" if USE_DRIVE else os.getcwd()
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
else:
    BACKEND = os.getcwd()       # local: this notebook sits in the repository folder
    OUTPUT_DIR = BACKEND
if not os.path.isfile(os.path.join(BACKEND, "objective.py")):
    raise RuntimeError(f"HW2 backend files not found in {BACKEND}. Locally, open this notebook "
                       "from inside the cloned repository folder.")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

# 2. Check the packages (Colab: install only what is missing or too old).
REQUIRED = {"jax": (0, 4, 30), "numpy": (1, 24), "matplotlib": (3, 7)}
def installed(package):
    try:
        return tuple(int(p) for p in re.findall(r"\d+", version(package))[:3])
    except PackageNotFoundError:
        return None
needed = [f"{p}>={'.'.join(map(str, v))}" for p, v in REQUIRED.items()
          if installed(p) is None or installed(p) < v]
if needed and not IN_COLAB:
    raise RuntimeError(f"Missing or too old: {needed}. In your activated virtual environment, "
                       "run: python -m pip install -r requirements.txt")
if needed:
    print("Installing", needed)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + needed, check=True)
    if "jax" in sys.modules or "numpy" in sys.modules:
        print("Packages were updated after being loaded: choose Runtime -> Restart session, "
              "then run this cell again.")

# 3. Work in OUTPUT_DIR: results/, submission/ and the ZIP are written there.
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(OUTPUT_DIR)
print(f"HW2 backend ready ({'Colab' if IN_COLAB else 'local'}): {BACKEND}")
print(f"Outputs are written to: {OUTPUT_DIR}")

In [ ]:
%matplotlib inline
import time

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

import _paths   # finds the msd_jax simulator folder and turns on double precision

print("Python    ", sys.version.split()[0])
print("NumPy     ", np.__version__)
print("JAX       ", jax.__version__, "| devices:", jax.devices())
print("Matplotlib", matplotlib.__version__)
print("double precision on:", jnp.zeros(1).dtype == jnp.float64)

SUBMISSION_DIR = "submission"           # figures and tables for your submission go here
os.makedirs(SUBMISSION_DIR, exist_ok=True)

## 3. Load the configuration

Every physical and numerical parameter lives in `config.py`, in one object called
`CONFIG`. Read it, but do not edit the file. To try a different value, make a
modified copy instead:

```python
from dataclasses import replace
my_config = replace(CONFIG, tip_load=0.8)
my_problem = ShapeProblem("cosine", my_config)
```

If you change `E_stiff`, `density` or the mesh, also set `dt=None` in the copy so the
time step is recomputed for stability.

In [ ]:
from dataclasses import replace
from config import CONFIG

C = CONFIG
print(f"Beam      : {C.n_cells_x} x {C.n_cells_y} cells, "
      f"{C.beam_length * 1e3:.0f} x {C.beam_height * 1e3:.0f} x {C.thickness * 1e3:.0f} mm")
print(f"Materials : E_soft = {C.E_soft / 1e6:g} MPa, E_stiff = {C.E_stiff / 1e6:g} MPa, "
      f"density = {C.density:g} kg/m^3")
print(f"Damping   : alpha = {C.alpha_damping:g} 1/s (drag on every node), "
      f"beta = {C.beta_damping:g} s (link damping c = beta * k)")
print(f"Load      : {C.tip_load:g} N downward at the free end, ramped up over {C.ramp_time:g} s, "
      f"simulated until T = {C.sim_time:g} s")
print(f"Time step : dt = {C.dt:g} s")
print(f"Targets   : A = {C.target_amplitude_ratio:g} x beam length, sigmoid beta = {C.sigmoid_beta:g}")
print(f"Fracture  : {C.fracture_n_cells_x} x {C.fracture_n_cells_y} cells, E = {C.fracture_E / 1e6:g} MPa, "
      f"load rate = {C.fracture_load_rate:g} N/s, F_fail = {C.fracture_force:g} N "
      f"({C.fracture_threshold_mode} threshold)")
print(f"Budget    : {C.eval_budget} function evaluations per run, {C.n_seeds} runs per method")

## 4. Build the cantilever beam

`ShapeProblem(target)` builds everything for one target shape: the lattice, the
clamp, the load, the time stepping, and the objective. The beam is 20 x 4 square
cells (5 mm each). Every cell has 4 side links (shared with its neighbours) and
2 diagonal links, and there is a lumped mass at every node.

* **Clamp:** all nodes on the left edge are fixed.
* **Load:** 1 N downward in total, shared by the nodes on the right edge.
* **Cells** are numbered `ix * 4 + iy` (`ix` along the length, `iy` up the
  height, both starting at 0), so a design is a vector of 80 numbers.
* **Centerline:** the row of **21 nodes at mid-height** (y = 10 mm). At the end of
  the simulation (t = T), the objective uses each node's **vertical displacement**,
  compared with the target evaluated at the node's **undeformed** position
  ξ = x₀ / L. Nodes also move up to ~1.5 mm sideways, so ξ is a label attached to the
  node, not its deformed x-position.

In [ ]:
from objective import ShapeProblem
from plotting import draw_links, draw_design, plot_design_result

problem = ShapeProblem("cosine")      # also builds the beam (the cosine target is just an example)
beam = problem.beam
print(problem.describe())
print(f"centerline: {len(beam.center_nodes)} nodes at y = {beam.y_center0 * 1e3:.0f} mm")

fig, ax = plt.subplots(figsize=(10, 3))
draw_links(ax, beam.nodes, beam,
           title="Undeformed beam: clamp (black squares), centerline nodes (blue), load (red)")
cx, cy = beam.nodes[beam.center_nodes].T
ax.plot(cx, cy, "o", color="tab:blue", ms=4)
F = np.asarray(beam.load_nodes)
for i in np.nonzero(F[:, 1])[0]:
    x, y = beam.nodes[i]
    length = 0.008 * abs(F[i, 1]) / np.abs(F[:, 1]).max()
    ax.annotate("", xy=(x + 0.004, y - length), xytext=(x + 0.004, y),
                arrowprops=dict(arrowstyle="->", color="tab:red"))
ax.set_xlim(-0.005, 0.11)
ax.set_ylim(-0.01, 0.025);

## 5. Fracture demo (worked example, not graded)

This section is complete; run it and read the output. It shows how the simulator
handles links that break.

* A uniform beam of 12 x 4 cells (E = 5 MPa) is clamped on the left. A downward
  force F(t) = load rate x t (20 N/s) pulls on its upper-right node.
* **Failure rule (force-based):** at every time step, a link fails when its elastic
  axial force exceeds the failure force, |k (L − L0)| > F_fail, with F_fail = 4 N
  for every link.
* A failed link gets activation a = 0, so it carries **no spring force and no
  damping force**. It stays in the arrays (array sizes never change) and failure
  is permanent.
* The optional `"stiffness"` threshold mode uses F_fail = F (k / k_ref)^p. With p = 1
  this is a fixed **elongation** |L − L0| > F / k_ref for every link. That is not a
  fixed strain, because diagonal links are √2 times longer than side links; a fixed
  strain criterion would need F_fail proportional to k·L0.

⏱ about 10-20 seconds.

In [ ]:
from fracture import run_fracture_demo
from plotting import plot_fracture_demo

frac = run_fracture_demo(CONFIG)
fbeam = frac["beam"]
print(f"Fracture beam: {fbeam.n_cells_x} x {fbeam.n_cells_y} cells, {fbeam.n_links} links, "
      f"E = {CONFIG.fracture_E / 1e6:g} MPa, load rate = {CONFIG.fracture_load_rate:g} N/s, "
      f"F_fail = {CONFIG.fracture_force:g} N")
if frac["first_failure_time"] is None:
    print("No link failed: raise fracture_load_rate or fracture_sim_time, or lower fracture_force.")
else:
    first = frac["first_links"]
    print(f"First failure time    : {frac['first_failure_time']:.4f} s")
    print(f"Load at first failure : {frac['first_failure_load']:.3f} N")
    print(f"First link(s) to fail : {first.tolist()} ({', '.join(fbeam.link_type[first])})")
    print(f"Number of failed links: {frac['n_failed']} of {fbeam.n_links}")
fig = plot_fracture_demo(frac, os.path.join(SUBMISSION_DIR, "fracture_demo.png"))

What to notice:

* The first link to fail is on the top edge at the clamp, where the bending
  moment is largest.
* The load is controlled (it keeps rising), so once one link fails the others
  around it are overloaded and fail within a few milliseconds: the crack runs
  through the root of the beam and the free part falls away.
* To experiment, pass a modified configuration, e.g.
  `run_fracture_demo(replace(CONFIG, fracture_force=6.0))`. Keep `fracture_E` at
  10 MPa or below: the time step was chosen for that stiffness.

## 6. Material assignment

Each cell has a modulus E. E is an **effective lattice modulus**: the number used to
set the spring stiffnesses, not measured data for a real material.

* Each cell contributes 3/8 · E · t to each of its 4 sides and 2 diagonals
  (t = thickness). A side shared by two cells gets both contributions.
* Link damping is c = β · k.
* Under a small, uniform strain this lattice reproduces the intended effective elastic
  response: an isotropic sheet with modulus E and Poisson ratio 1/3. It is still a
  coarse spring lattice (4 cells high), not a continuum finite-element model; for
  example, in bending it is about 8% stiffer than a continuum beam with the same E.

**Design vectors** have one entry per cell:

* binary design `b`: 0 = soft (1 MPa), 1 = stiff (10 MPa), used by random search and the GA
* continuous design `z` in [0, 1]: E = E_soft + z (E_stiff − E_soft), used by the gradient method

Always keep z inside [0, 1] (`z = np.clip(z, 0, 1)`). Values outside would mean a
modulus outside the soft-stiff range, and z > 1 can make the explicit time
stepping unstable.

In [ ]:
from materials import design_to_modulus, material_to_link_stiffness, link_damping

n = problem.n_design                    # 80 design variables, one per cell
all_soft = np.zeros(n)
cell0_stiff = np.zeros(n)
cell0_stiff[0] = 1.0                    # cell 0 = bottom cell at the clamp

for name, design in [("all soft", all_soft), ("cell 0 stiff", cell0_stiff)]:
    E = design_to_modulus(design, CONFIG)        # [Pa], one value per cell
    k = material_to_link_stiffness(E, beam)      # [N/m], one value per link
    c = link_damping(k, CONFIG)                  # [N s/m], c = beta * k
    print(f"{name:12s}: E in [{E.min() / 1e6:g}, {E.max() / 1e6:g}] MPa, "
          f"k in [{k.min():.0f}, {k.max():.0f}] N/m, c in [{c.min():.1e}, {c.max():.1e}] N s/m")

# The 6 links that belong to cell 0, and how making it stiff changes them:
k_soft = np.asarray(material_to_link_stiffness(design_to_modulus(all_soft, CONFIG), beam))
k_one = np.asarray(material_to_link_stiffness(design_to_modulus(cell0_stiff, CONFIG), beam))
for link in np.nonzero(np.asarray(beam.cell_to_link)[:, 0])[0]:
    print(f"  link {link:3d} ({beam.link_type[link]:10s}): k = {k_soft[link]:6.0f} -> {k_one[link]:6.0f} N/m")

`problem.link_properties(design)` does all three steps at once and returns `(k, c)`.
Below is an arbitrary example design (a checkerboard, not a good answer) and the
link stiffness it produces.

In [ ]:
checkerboard = np.array([(ix + iy) % 2 for ix in range(beam.n_cells_x)
                         for iy in range(beam.n_cells_y)], dtype=float)   # index = ix * 4 + iy
k_ex, c_ex = problem.link_properties(checkerboard)

fig, axes = plt.subplots(2, 1, figsize=(9, 4.5))
draw_design(axes[0], beam, checkerboard, "checkerboard design (dark = stiff)")
draw_links(axes[1], beam.nodes, beam, k=k_ex, title="resulting link stiffness (darker = stiffer)")
fig.tight_layout()
fig.savefig(os.path.join(SUBMISSION_DIR, "material_demo.png"), dpi=120)

## 7. The three target profiles

With ξ = (x − x_min) / (x_max − x_min), A = 0.15 L = 15 mm and β = 8:

* linear: y = −A ξ
* cosine: y = −(A/2)(1 − cos πξ)
* sigmoid: s(ξ) = 1 / (1 + exp(−β(ξ − 0.5))), and y = −A (s(ξ) − s(0)) / (s(1) − s(0))

All three start at 0 at the clamp and end at −A at the tip.

**None of the targets is expected to be matched exactly.** Under a single tip
load the bending moment has the same sign along the whole beam, so (in beam
theory) the curvature of the centerline cannot change sign. The cosine and the
sigmoid both have an inflection point at ξ = 0.5, and the linear target needs a
non-zero slope at the clamp. The sigmoid is the hardest of the three. Judge the
optimization methods against one another, not against zero error.

In [ ]:
from target_profiles import get_target

xi = np.linspace(0, 1, 201)
A = problem.A
fig, ax = plt.subplots(figsize=(7, 4))
for name in ["linear", "cosine", "sigmoid"]:
    ax.plot(xi, 1e3 * np.asarray(get_target(name, xi, A, CONFIG.sigmoid_beta)), label=name)
ax.plot(beam.xi, 1e3 * np.asarray(problem.y_target), "k.", label="cosine at the 21 centerline nodes")
ax.set_xlabel("ξ = normalized position along the beam")
ax.set_ylabel("target vertical displacement [mm]")
ax.set_title(f"Targets (A = {A * 1e3:.0f} mm, β = {CONFIG.sigmoid_beta:g})")
ax.grid(alpha=0.3)
ax.legend()
fig.savefig(os.path.join(SUBMISSION_DIR, "targets.png"), dpi=120);

## 8. The objective

$$J = \frac{1}{21}\sum_{\text{centerline nodes}} \big(y_\text{centerline} - y_\text{target}\big)^2 \quad [\text{m}^2]$$

`problem.rms_error_ratio(J)` = √J / A is easier to read: the RMS centerline error as
a fraction of the tip target.

**Function-evaluation convention** (identical for every method):

| call | counts as |
| --- | --- |
| `problem.evaluate(design)` | 1 evaluation |
| `problem.evaluate_batch(designs)` | 1 per design (row) |
| `problem.value_and_grad(z)` | 1 evaluation (it also returns dJ/dz) |

Every call is logged in `problem.log` as `(J, is_binary_design, wall_time)`, and
`problem.reset_counter()` starts a new run. The convergence plots use the **best J
of a binary (buildable) design** so far; a continuous design uses up an evaluation
but cannot set the best-so-far value. Wall-clock time is logged too, because a
gradient costs several times more than a plain evaluation.

Always run the simulator through these methods inside your optimizers, so the
counting is right.

In [ ]:
problem.reset_counter()
rng = np.random.default_rng(0)
examples = {"all soft": np.zeros(n), "all stiff": np.ones(n),
            "random": (rng.random(n) < 0.5).astype(float)}
for name, design in examples.items():
    t0 = time.perf_counter()
    J = problem.evaluate(design)
    print(f"{name:9s}: J = {J:.3e} m^2, RMS error = {100 * problem.rms_error_ratio(J):5.1f}% of A "
          f"({time.perf_counter() - t0:.2f} s)")
print("evaluations used:", problem.n_evals)
print("last log entry (J, is_binary_design, wall_time):", problem.log[-1])
fig = plot_design_result(problem, examples["random"], title="A random design on the cosine target")

In [ ]:
z = np.full(n, 0.5)                     # a continuous ("gray") design
J, dJdz = problem.value_and_grad(z)     # first call compiles the gradient (a few seconds)
print(f"z = 0.5 everywhere: J = {J:.3e}, dJ/dz has {dJdz.size} entries "
      f"in [{dJdz.min():.2e}, {dJdz.max():.2e}]")
print("evaluations used:", problem.n_evals, "| was the last one a binary design?", problem.log[-1][1])

t0 = time.perf_counter(); problem.evaluate(examples["random"]); t_eval = time.perf_counter() - t0
t0 = time.perf_counter(); problem.value_and_grad(z);           t_grad = time.perf_counter() - t0
print(f"one evaluate: {1e3 * t_eval:.0f} ms, one value_and_grad: {1e3 * t_grad:.0f} ms "
      f"(each counts as 1 evaluation)")

## 9. Random search ✏️ STUDENT TODO

Write `random_search`: draw random binary designs (each cell stiff with probability
`p_stiff`), evaluate each one with `problem.evaluate`, and keep the best. Use the
`rng` that the function creates, so a given seed always gives the same run.

Keep the function name, the arguments and the returned dictionary keys: the
checks and the convergence experiments rely on them. You may add helper functions.

After editing, run the cell (it only defines the function), then run the next cell
to try it and check it.

In [ ]:
def random_search(problem, max_evals, seed=0, p_stiff=0.5):
    """Evaluate `max_evals` random binary designs and return the best.

    Inputs
        problem   : objective.ShapeProblem (use problem.evaluate(design))
        max_evals : number of function evaluations to spend
        seed      : random seed, so runs are reproducible
        p_stiff   : probability that each design entry is stiff
    Returns a dict with
        "best_design" : best binary design found (array of 0/1)
        "best_J"      : its objective value
        "log"         : problem.log, one entry per evaluation (for convergence plots)
    """
    rng = np.random.default_rng(seed)
    problem.reset_counter()
    best_design, best_J = None, np.inf

    # STUDENT TODO: draw random binary designs with rng, evaluate each one with
    # problem.evaluate(design), and keep track of the best design and its J.
    raise NotImplementedError("TODO: complete random_search (notebook section 9)")

    return {"best_design": best_design, "best_J": best_J, "log": list(problem.log)}

In [ ]:
from run_all_checks import run_check, check_random_search

try:
    rs = random_search(problem, max_evals=50, seed=0)
    print(f"random search, 50 evaluations: best J = {rs['best_J']:.3e} "
          f"(RMS error {100 * problem.rms_error_ratio(rs['best_J']):.1f}% of A)")
    fig = plot_design_result(problem, rs["best_design"], title="random search, 50 evaluations")
except NotImplementedError as e:
    print(e)
run_check("random search", check_random_search, random_search, problem);

## 10. Evolutionary optimization (genetic algorithm) ✏️ STUDENT TODO

Genome: one bit per cell (0 = soft, 1 = stiff). Each generation:

1. **elitism**: copy the `n_elite` best individuals unchanged
2. **selection**: pick parents by tournament (the best of `tournament_size` random
   picks), using the provided `tournament_select`
3. **crossover**: uniform crossover with probability `p_crossover`
4. **mutation**: flip each bit with probability `p_mutation` (default 1 / number of bits)
5. **evaluate** the new children (elites keep their known J)

Stop when `max_evals` evaluations have been used (never more). The initial random
population is already written for you. The first cell below is a provided helper;
the second is yours.

In [ ]:
def tournament_select(rng, fitness, tournament_size):
    """Index of the best (lowest J) of `tournament_size` randomly chosen individuals."""
    picks = rng.integers(0, len(fitness), size=tournament_size)
    return picks[np.argmin(fitness[picks])]

In [ ]:
def genetic_algorithm(problem, max_evals, seed=0, pop_size=40, n_elite=2,
                      tournament_size=3, p_crossover=0.9, p_mutation=None):
    """Minimize problem.evaluate over binary designs with a GA.

    Returns a dict with
        "best_design" : best binary design found
        "best_J"      : its objective value
        "log"         : problem.log, one entry per evaluation
    """
    rng = np.random.default_rng(seed)
    problem.reset_counter()
    n = problem.n_design
    if p_mutation is None:
        p_mutation = 1.0 / n

    # Initial population: random binary designs.
    population = (rng.random((pop_size, n)) < 0.5).astype(float)
    fitness = problem.evaluate_batch(population)

    # STUDENT TODO: write the generation loop (elitism, tournament selection,
    # crossover, mutation, evaluation of children). Stop at max_evals.
    raise NotImplementedError("TODO: complete genetic_algorithm (notebook section 10)")

    best = int(np.argmin(fitness))
    return {"best_design": population[best], "best_J": float(fitness[best]),
            "log": list(problem.log)}

In [ ]:
from run_all_checks import check_genetic_algorithm

try:
    ga = genetic_algorithm(problem, max_evals=200, seed=0, pop_size=20)
    print(f"GA, 200 evaluations: best J = {ga['best_J']:.3e} "
          f"(RMS error {100 * problem.rms_error_ratio(ga['best_J']):.1f}% of A)")
except NotImplementedError as e:
    print(e)
run_check("genetic algorithm", check_genetic_algorithm, genetic_algorithm, problem);

## 11. Gradient-based optimization ✏️ STUDENT TODO

The design is continuous, z in [0, 1] per cell, with E = E_soft + z (E_stiff − E_soft).
`problem.value_and_grad(z)` returns J and dJ/dz, computed by JAX automatic
differentiation through all ~6250 time steps.

Each iteration should cost **2 function evaluations**:

1. `J, g = problem.value_and_grad(z)`: the continuous design, gives the gradient
2. `problem.evaluate(threshold_design(z))`: the binary design you would actually
   build (z ≥ 0.5 → stiff); remember the best one

Then take a gradient step (plain gradient descent or Adam) and **clip z to [0, 1]**.

The first cell below provides an optional helper, `projection`. Read its docstring
after you have looked at what thresholding does to your designs (section 12).

In [ ]:
from materials import threshold_design


def projection(z, sharpness, eta=0.5):
    """Smoothed Heaviside step (tanh projection). Maps [0,1] -> [0,1]."""
    a = jnp.tanh(sharpness * eta)
    return (a + jnp.tanh(sharpness * (z - eta))) / (a + jnp.tanh(sharpness * (1.0 - eta)))


def projected_value_and_grad(problem, z, sharpness):
    """J and dJ/dz when the simulation uses z_phys = projection(z, sharpness).

    Counts as 1 function evaluation (it is one value_and_grad call). The chain
    rule factor dz_phys/dz is applied by hand so problem.value_and_grad is reused.
    """
    z_phys, dproj = jax.jvp(lambda u: projection(u, sharpness), (z,), (np.ones_like(z),))
    J, g_phys = problem.value_and_grad(np.asarray(z_phys))
    return J, g_phys * np.asarray(dproj), np.asarray(z_phys)

In [ ]:
def gradient_optimize(problem, max_evals, seed=0, lr=0.05, z0=None,
                      projection_on=True, sharpness_schedule=(1, 2, 4, 8, 16, 32),
                      beta1=0.9, beta2=0.999, eps=1e-12):
    """Minimize J over a continuous design z in [0, 1] using gradients.

    Inputs
        problem   : objective.ShapeProblem
        max_evals : evaluation budget (2 per iteration, see module docstring)
        seed      : seed for the random initial design (used if z0 is None)
        lr        : Adam step size in units of z
        z0        : optional initial design; default uniform random in [0, 1]
        projection_on      : use the projection continuation (see module docstring)
        sharpness_schedule : projection sharpness values, used for equal
                             shares of the iterations
    Returns a dict with
        "z"                      final continuous design
        "J_continuous"           J(z) of the final continuous design
        "best_design", "best_J"  best thresholded (binary) design seen and its J
        "history_continuous"     J(z) at every iteration
        "log"                    problem.log
    """
    rng = np.random.default_rng(seed)
    problem.reset_counter()
    z = rng.random(problem.n_design) if z0 is None else np.array(z0, dtype=float)
    best_design, best_J = None, np.inf
    history_continuous = []

    # STUDENT TODO: implement the optimization loop. Each iteration should
    #   1. get J, dJ/dz = problem.value_and_grad(z)
    #   2. evaluate the thresholded design and remember the best binary design
    #   3. take a gradient step (plain gradient descent or Adam) and clip z to [0, 1]
    raise NotImplementedError("TODO: complete gradient_optimize (notebook section 11)")

    if projection_on:
        z = np.asarray(projection(z, sharpness_schedule[-1]))
    J_final = float(problem.objective(z))   # diagnostic only, not counted
    return {"z": z, "J_continuous": J_final, "best_design": best_design,
            "best_J": best_J, "history_continuous": history_continuous,
            "log": list(problem.log)}

In [ ]:
from run_all_checks import check_gradient_optimize

run_check("gradient method", check_gradient_optimize, gradient_optimize, problem);

## 12. Threshold the gradient design

The gradient method works with gray values of z, but only a soft/stiff design can be
built. Here you compare the continuous design with its thresholded version
(z ≥ 0.5 → stiff, z < 0.5 → soft).

* How many cells are still gray (far from 0 or 1)?
* How much worse is J after thresholding?
* If your `gradient_optimize` has options (for example, projection on or off),
  compare them here.

⏱ about 10-30 seconds.

In [ ]:
try:
    gr = gradient_optimize(problem, max_evals=100, seed=0)
except NotImplementedError as e:
    gr = None
    print(e)

if gr is not None:
    z_final = np.asarray(gr["z"])
    b_final = np.asarray(threshold_design(z_final))
    J_threshold = float(problem.objective(b_final))   # diagnostic only, not a counted evaluation
    gray = np.mean((z_final > 0.05) & (z_final < 0.95))
    print(f"continuous design z     : J = {gr['J_continuous']:.3e} "
          f"(RMS {100 * problem.rms_error_ratio(gr['J_continuous']):.1f}% of A), {100 * gray:.0f}% of cells gray")
    print(f"thresholded final z     : J = {J_threshold:.3e} (RMS {100 * problem.rms_error_ratio(J_threshold):.1f}% of A)")
    print(f"best binary design seen : J = {gr['best_J']:.3e} (RMS {100 * problem.rms_error_ratio(gr['best_J']):.1f}% of A)")

    fig, axes = plt.subplots(3, 1, figsize=(8, 6))
    draw_design(axes[0], beam, z_final, "final continuous design z")
    draw_design(axes[1], beam, b_final, "thresholded design (z >= 0.5 -> stiff)")
    axes[2].hist(z_final, bins=20, range=(0, 1))
    axes[2].set_xlabel("z")
    axes[2].set_ylabel("number of cells")
    fig.tight_layout()

## 13. Repeated trials

`run_trials(method, target, n_seeds, budget, problem=...)` runs your optimizer with
seeds 0, 1, 2, … and records, for every run, the **best-so-far J of a binary design**
after each function evaluation.

**Settings** (only the number of runs and evaluations change; the physics is the same):

* `QUICK_MODE = True` (default): 2 runs x 100 evaluations, **for debugging only**.
  ⏱ about 2-5 minutes for all three targets.
* `QUICK_MODE = False`: the assignment setting, 10 runs x 1000 evaluations
  per method and target. ⏱ One run of all three methods takes about 2 minutes
  on a laptop and 3.5 minutes on free Colab, so one target takes about
  20 minutes on a laptop and 35 minutes on Colab.
  Plan one target per sitting.

**One target at a time, and resuming.** The targets run one after another (linear,
cosine, sigmoid). After each method finishes, its results are saved immediately in
`results/` (quick mode: `results_quick/`), and after each target its plots go to
`submission/`. When you run this cell again, everything that is already saved is
**skipped**, so after a disconnect you can just run the notebook again and it
continues where it stopped. Set `TARGETS_TO_RUN = ["linear"]` to run only one
target this session. Set `FORCE_RERUN = True` only if you changed your optimizer
and want to recompute everything.

> On Colab, saved results survive a disconnect **only** with `USE_DRIVE = True`
> (section 2). Otherwise the files are lost when the runtime is recycled; download
> the ZIP (section 15) after each target as a backup.

> 💾 **Before starting full experiments, make sure this notebook is saved in your
> Google Drive** (*File → Save a copy in Drive*, then *File → Save*).

In [ ]:
import gc
from convergence import (run_trials, save_results, has_results, load_results, plot_convergence,
                         save_target_outputs, outputs_up_to_date, summary_table)

QUICK_MODE = True         # False = assignment setting (10 runs x 1000 evaluations)
RUN_EXPERIMENTS = True    # False = do not run anything, only use results saved earlier
FORCE_RERUN = False       # True = recompute even if results are already saved
TARGETS_TO_RUN = ["linear", "cosine", "sigmoid"]   # e.g. ["linear"] for one target per session

if QUICK_MODE:
    N_SEEDS, BUDGET, RESULTS_DIR = 2, 100, "results_quick"
else:
    N_SEEDS, BUDGET, RESULTS_DIR = CONFIG.n_seeds, CONFIG.eval_budget, "results"

TARGETS = ["linear", "cosine", "sigmoid"]
METHODS = {"random": random_search, "genetic": genetic_algorithm, "gradient": gradient_optimize}
print(f"{'QUICK (debugging)' if QUICK_MODE else 'FULL (assignment)'} mode: "
      f"{N_SEEDS} runs x {BUDGET} evaluations per method and target, saved in {RESULTS_DIR}/")

In [ ]:
for target in TARGETS:                      # strictly one target after another
    if target not in TARGETS_TO_RUN:
        continue
    to_run = [m for m in METHODS
              if FORCE_RERUN or not has_results(target, m, N_SEEDS, BUDGET, RESULTS_DIR)]
    if not RUN_EXPERIMENTS:
        to_run = []
    if not to_run:
        if not load_results(target, RESULTS_DIR):
            print(f"{target}: no saved results in {RESULTS_DIR}/ and RUN_EXPERIMENTS = False")
            continue
        if outputs_up_to_date(target, RESULTS_DIR, SUBMISSION_DIR):
            print(f"{target}: already done (results in {RESULTS_DIR}/, plots in {SUBMISSION_DIR}/)")
            continue
        print(f"{target}: already done, making its plots from {RESULTS_DIR}/")
    # Reuse the cosine problem from section 4; build the others when they are needed.
    target_problem = problem if target == "cosine" else ShapeProblem(target)
    for method_name in to_run:
        try:
            data = run_trials(METHODS[method_name], target, N_SEEDS, BUDGET,
                              problem=target_problem, method_name=method_name)
        except NotImplementedError as e:
            print(f"  {target:8s} {method_name:8s} skipped: {e}")
            continue
        save_results(target, method_name, data, RESULTS_DIR)   # saved right away
        del data
    if save_target_outputs(target, target_problem, RESULTS_DIR, SUBMISSION_DIR):
        print(f"{target}: plots and results written to {SUBMISSION_DIR}/")
    if target_problem is not problem:
        del target_problem                  # free the compiled simulator before the next target
    gc.collect()
    jax.clear_caches()

## 14. Convergence comparison

One plot per target, all three methods on it. The x-axis is function evaluations;
the y-axis is the best-so-far J of a binary design (log scale).

* Solid line: the **mean** over the runs.
* Shaded region: the **2.5–97.5 percentile band** of the runs at each evaluation
  count (the central 95% of the runs; with 10 runs this is close to the min–max
  range). It shows how much one run can differ from another. It is **not** a 95%
  confidence interval for the mean, which would be much narrower.

The table under each plot gives the final RMS error / A per method: mean, standard
deviation, 2.5 and 97.5 percentiles, best and worst run, and wall time per run.

In [ ]:
A = CONFIG.target_amplitude_ratio * CONFIG.beam_length
results = {target: load_results(target, RESULTS_DIR) for target in TARGETS}
for target in TARGETS:
    if not results[target]:
        print(f"{target}: no saved results in {RESULTS_DIR}/ yet")
        continue
    fig = plot_convergence(results[target], target,
                           os.path.join(SUBMISSION_DIR, f"convergence_{target}.png"))
    plt.show()
    print(summary_table(results[target], target, A), "\n")

## 15. Final checklist and submission ZIP

Before you submit:

- [ ] Sections 9, 10 and 11 print `[PASS]` (re-run the check cells, or the cell below).
- [ ] Section 13 was run with `QUICK_MODE = False` for all three targets.
- [ ] The convergence plots (section 14) show all three methods for each target.
- [ ] *Restart and Run All* runs without errors (with your full results saved,
      section 13 skips the finished work, so this is fast).
- [ ] You downloaded `HW2_submission.zip` (cell below) **and** your notebook
      (*File → Download → Download .ipynb*).
- [ ] Your written answers (see the assignment PDF) refer to the figures in the ZIP.

The `submission/` folder collects: `fracture_demo.png`, `material_demo.png`,
`targets.png`, `convergence_<target>.png`, `best_<target>_<method>.png` (best design of
each method), `summary.csv` (the tables of section 14), and `results/` (your raw run
data).

In [ ]:
from run_all_checks import check_convergence

statuses = [run_check("random search", check_random_search, random_search, problem),
            run_check("genetic algorithm", check_genetic_algorithm, genetic_algorithm, problem),
            run_check("gradient method", check_gradient_optimize, gradient_optimize, problem),
            run_check("repeated runs", check_convergence, random_search, problem)]
print("all optimizer checks pass" if all(s == "PASS" for s in statuses) else "some checks are not passing yet")

In [ ]:
import shutil
from convergence import write_summary_csv

done = {target: results[target] for target in TARGETS if results[target]}
if done:
    write_summary_csv(done, A, os.path.join(SUBMISSION_DIR, "summary.csv"))
with open(os.path.join(SUBMISSION_DIR, "README.txt"), "w") as f:
    f.write(f"{'QUICK (debugging)' if QUICK_MODE else 'FULL (assignment)'} mode: "
            f"{N_SEEDS} runs x {BUDGET} evaluations per method\n"
            f"targets with results: {', '.join(done) or 'none'}\n")

zip_path = shutil.make_archive("HW2_submission", "zip", root_dir=".", base_dir=SUBMISSION_DIR)
print(f"Wrote {os.path.abspath(zip_path)} with:")
for root, _, files in sorted(os.walk(SUBMISSION_DIR)):
    for name in sorted(files):
        print("  ", os.path.relpath(os.path.join(root, name), SUBMISSION_DIR))
if QUICK_MODE:
    print("NOTE: these are QUICK-mode (debugging) results. Set QUICK_MODE = False in section 13 "
          "for your submission.")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)        # starts the download in your browser